# Module 2: RAG & Codebase Context Retrieval

**Course:** UNDP IICPSD Agentic AI & Vibe Coding Training Series 2026 | **Instructor:** Phyo Ko Ko, Senior Cyber Security Analyst

---

## Why This Module Matters

When you ask an AI agent to modify a function in your codebase, the LLM has no built-in knowledge of your files. Without RAG, it either **hallucinates** code that does not match your project, or you must paste your entire codebase into the prompt - which is expensive and often exceeds the context window limit.

**RAG (Retrieval-Augmented Generation)** solves this: the IDE indexes your codebase into vector embeddings, your prompt triggers a semantic search, and only the most relevant snippets are injected into the LLM context window.

```
Your Prompt
    |
    v
[Embedding Model]  <-- converts your prompt to a vector
    |
    v
[Vector Search]  <-- finds the closest matching code snippets
    |
    v
[Context Injection]  <-- injects only relevant snippets into LLM prompt
    |
    v
[LLM Response]  <-- grounded in your actual codebase, not guesswork
```

## The FinOps Angle

Injecting only the relevant snippets (instead of the whole codebase) dramatically reduces token usage. At enterprise scale - thousands of prompts per day - this difference in cost is significant. Mastering RAG is therefore both a quality and a financial optimization.


## Key Terms

| Term | Definition |
|---|---|
| **RAG** | Retrieval-Augmented Generation - retrieves relevant external context and injects it into the LLM prompt before generation. |
| **Embedding** | A numerical vector representation of text that captures semantic meaning. Similar text produces similar vectors. |
| **Vector Search** | Finding the most semantically similar embeddings to a query vector - the core of RAG retrieval. |
| **Context Window** | The maximum tokens an LLM can process in one request - typically 8k to 200k tokens depending on the model. |
| **Hallucination** | When an LLM generates confident but factually incorrect output due to lack of grounding in real context. |
| **FinOps** | Financial operations for cloud and API costs - optimizing token usage directly reduces API spend. |


---

## Lab 2: Codebase Retrieval-Augmented Generation (RAG)

### Objective

Simulate a lightweight RAG pipeline that retrieves only the relevant files from a mock codebase based on user intent. This mirrors what Cursor does behind the scenes when you type a prompt in the editor.


In [ ]:
mock_codebase = {
    "auth.py": "def login_user(username, password):\n    print('Logging in user...')\n    return True",
    "db.py": "def connect_database():\n    db_uri = 'postgresql://localhost:5432/main_db'\n    return db_uri",
    "weather.py": "import requests\ndef get_current_weather(city):\n    return f'Weather data for {city}'"
}

def codebase_rag_retriever(user_intent):
    print(f"User Intent: '{user_intent}'")
    print("Searching codebase...")

    retrieved_context = ""
    for filename, code in mock_codebase.items():
        keyword = user_intent.lower().split()[-1]
        if keyword in filename or keyword in code:
            print(f"  Match found: {filename}")
            retrieved_context += f"\n--- Context from {filename} ---\n{code}\n"

    if not retrieved_context:
        retrieved_context = "No relevant context found in codebase."

    return retrieved_context

user_prompt = "Refactor the logic inside my database connection script"
augmented_context = codebase_rag_retriever(user_prompt)

print("\n--- Context Passed to LLM Prompt Window ---")
print(augmented_context)


## What Just Happened?

The retriever found `db.py` because the last keyword `'script'` matched content inside the file. In a real IDE like Cursor, this search uses **semantic vector similarity** instead of keyword matching - meaning it would find `db.py` even if the user said `'fix the connection handler'` without mentioning the filename.

The retrieved snippet is then **injected into the hidden background prompt** that the LLM receives - giving it real context about your codebase before it writes a single line of code.


---

## Lab 2b: Context Engineering — Grounding the Agent

### Prompt Engineering vs Context Engineering

| | Prompt Engineering | Context Engineering |
|---|---|---|
| **Focus** | How you word the instruction | What information surrounds the instruction |
| **Lever** | The user message | System prompt + retrieved context + history |
| **Goal** | Clear task description | Full situational awareness for the agent |
| **Analogy** | Asking a good question | Briefing someone before they answer |

> Context engineering is the practice of **designing the full information environment** an LLM receives — system prompt, retrieved documents, conversation history, tool outputs — so it can reason correctly without hallucinating.

### The Four Layers of Context

```
┌─────────────────────────────────────────┐
│  1. SYSTEM PROMPT                       │  ← Role, rules, output format
│     "You are a security analyst..."     │
├─────────────────────────────────────────┤
│  2. RETRIEVED CONTEXT (RAG)             │  ← Relevant codebase snippets
│     --- auth.py ---                     │
│     def login_user(...)                 │
├─────────────────────────────────────────┤
│  3. CONVERSATION HISTORY                │  ← What was said before
│     User: Fix the login bug             │
│     Agent: I changed line 12...         │
├─────────────────────────────────────────┤
│  4. USER MESSAGE                        │  ← The actual instruction
│     "Now add rate limiting"             │
└─────────────────────────────────────────┘
```

### Objective

Build a context assembler that combines all four layers into a single grounded prompt, and observe how the quality of the assembled context affects agent reasoning.

In [ ]:
# ============================================================
# Lab 2b: Context Engineering — Grounding the Agent
# ============================================================

def assemble_context(system_prompt, retrieved_docs, history, user_message):
    """Assembles all 4 context layers into a single grounded prompt string."""
    sections = []

    # Layer 1: System Prompt
    sections.append(f"<system>\n{system_prompt.strip()}\n</system>")

    # Layer 2: Retrieved Context
    if retrieved_docs:
        doc_block = "\n".join(
            f"--- {doc['name']} ---\n{doc['content']}"
            for doc in retrieved_docs
        )
        sections.append(f"<context>\n{doc_block}\n</context>")
    else:
        sections.append("<context>\nNo relevant context retrieved.\n</context>")

    # Layer 3: Conversation History
    if history:
        history_block = "\n".join(
            f"{turn['role'].upper()}: {turn['content']}" for turn in history
        )
        sections.append(f"<history>\n{history_block}\n</history>")

    # Layer 4: User Message
    sections.append(f"<user>\n{user_message.strip()}\n</user>")

    return "\n\n".join(sections)


def token_estimate(text):
    """Rough token estimate: ~4 characters per token."""
    return len(text) // 4


# ─── Scenario A: No context (baseline — agent is flying blind) ───────────────
print("=" * 60)
print("SCENARIO A: No context (baseline)")
print("=" * 60)

prompt_a = assemble_context(
    system_prompt="You are a helpful coding assistant.",
    retrieved_docs=[],
    history=[],
    user_message="Fix the login bug."
)
print(prompt_a)
print(f"\n[Tokens ~{token_estimate(prompt_a)}]")
print("\n[Agent reasoning]: Which login? Which bug? Likely to hallucinate.")


# ─── Scenario B: With system prompt + RAG context ────────────────────────────
print("\n" + "=" * 60)
print("SCENARIO B: System prompt + RAG context")
print("=" * 60)

retrieved = [
    {
        "name": "auth.py",
        "content": (
            "def login_user(username, password):\n"
            "    # BUG: no rate limiting, brute-force possible\n"
            "    if username == 'admin' and password == 'admin123':\n"
            "        return True\n"
            "    return False"
        )
    }
]

prompt_b = assemble_context(
    system_prompt=(
        "You are a senior security engineer reviewing Python code. "
        "Identify security issues and propose fixes. "
        "Output a corrected function with inline comments explaining each change."
    ),
    retrieved_docs=retrieved,
    history=[],
    user_message="Fix the login bug in auth.py."
)
print(prompt_b)
print(f"\n[Tokens ~{token_estimate(prompt_b)}]")
print("\n[Agent reasoning]: Grounded — knows the exact file, sees the hardcoded credentials and missing rate limit.")


# ─── Scenario C: Full 4-layer context ────────────────────────────────────────
print("\n" + "=" * 60)
print("SCENARIO C: All 4 layers — maximum grounding")
print("=" * 60)

history = [
    {"role": "user",    "content": "I need to harden the authentication module."},
    {"role": "assistant","content": "Understood. I'll review auth.py for credential hygiene, rate limiting, and input validation."},
]

prompt_c = assemble_context(
    system_prompt=(
        "You are a senior security engineer reviewing Python code. "
        "Identify security issues and propose fixes. "
        "Output a corrected function with inline comments explaining each change."
    ),
    retrieved_docs=retrieved,
    history=history,
    user_message="Now add rate limiting to prevent brute-force attacks."
)
print(prompt_c)
print(f"\n[Tokens ~{token_estimate(prompt_c)}]")
print("\n[Agent reasoning]: Fully grounded — knows the role, sees the code, remembers the prior discussion, understands the exact next step.")


# ─── Token budget comparison ──────────────────────────────────────────────────
print("\n" + "=" * 60)
print("TOKEN BUDGET COMPARISON")
print("=" * 60)
for label, prompt in [("A: No context", prompt_a), ("B: System+RAG", prompt_b), ("C: Full 4-layer", prompt_c)]:
    print(f"  {label:20} ~{token_estimate(prompt):>4} tokens")
print("\nConclusion: Scenario C adds ~history overhead but dramatically improves grounding.")
print("Scenario A is cheapest but gives the agent nothing to reason with — quality collapses.")

## What Just Happened? (Lab 2b)

1. **Scenario A** — no context: the agent receives only `"Fix the login bug."` It has no idea which file, which bug, or what the codebase looks like. This is how most beginners use AI tools — and why they get hallucinated, generic code.

2. **Scenario B** — system prompt + RAG: the agent now knows its role (security engineer), sees the actual buggy code, and can reason about the specific hardcoded credentials and missing rate limiting.

3. **Scenario C** — all 4 layers: the agent additionally knows the conversation history, so it understands that hardening authentication is the broader goal and that rate limiting is the agreed next step. This produces the most precise, context-aware response.

4. **Token budget**: Full context costs more tokens, but RAG (from Lab 2) keeps it efficient by injecting only the *relevant* files — not the entire codebase.

> **Key insight:** The gap between a good AI response and a hallucinated one is almost never the model — it is the quality of the context you assembled around the instruction. Context engineering is the discipline of closing that gap systematically.

---

## Assignment 2

### Part A: Reflection Questions

**Q1.** Without RAG, what problem does an LLM face when asked to modify a specific function in your codebase? What would it likely do instead?

> *Your answer here...*

**Q2.** The Lab 2 retriever uses only the last word of the user intent as the search keyword. Give two real-world scenarios where this approach would fail completely.

> *Your answer here...*

**Q3.** What does context window mean, and why is it important to inject only the most relevant snippets rather than the entire codebase?

> *Your answer here...*

### Part B: Coding Exercise - Expand the Codebase & Improve Retrieval

1. Add **3 more realistic files** to `mock_codebase` (e.g. `payments.py`, `notifications.py`, `reports.py`)
2. **Improve the retrieval logic** to search ALL words in the user intent, not just the last one
3. **Test with 2 different prompts** and verify the correct files are retrieved

### Part D: Context Engineering Exercise

**Q4.** In Scenario A, the agent received only the user message. Name two specific things it would likely get wrong in its response, and explain why.

> *Your answer here...*

**Q5.** You are building an agent that helps UNDP staff draft policy briefs. Design the system prompt (Layer 1) for this agent. What role should it be given, what rules should it follow, and what output format should it produce?

> *Your answer here...*

**Q6.** Your context assembler currently puts history before the user message. A colleague suggests moving history after the user message to "keep the instruction fresh." Who is right, and why does order matter in the context window?

> *Your answer here...*

In [ ]:
# TODO 1: Add 3 more files to the codebase
mock_codebase = {
    "auth.py": "def login_user(username, password):\n    print('Logging in user...')\n    return True",
    "db.py": "def connect_database():\n    db_uri = 'postgresql://localhost:5432/main_db'\n    return db_uri",
    "weather.py": "import requests\ndef get_current_weather(city):\n    return f'Weather data for {city}'",
    # Add your 3 new files here:
}

# TODO 2: Improve retrieval to match ANY keyword in the intent
def improved_rag_retriever(user_intent):
    print(f"User Intent: '{user_intent}'")
    keywords = user_intent.lower().split()  # all words
    retrieved_context = ""

    for filename, code in mock_codebase.items():
        keyword = keywords[-1]  # TODO: change this to check ALL keywords, not just the last one
        if keyword in filename or keyword in code:
            print(f"  Match: {filename}")
            retrieved_context += f"\n--- {filename} ---\n{code}\n"

    return retrieved_context if retrieved_context else "No relevant context found."

# TODO 3: Test with 2 prompts
print("=== Test 1 ===")
print(improved_rag_retriever("Fix the login authentication bug"))

print("\n=== Test 2 ===")
print(improved_rag_retriever("your second test prompt here"))
